# Objetivo_4

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import altair as alt
import plotly.graph_objects as go
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
df = pd.read_excel('datos_CAIMEDv8_unido_v5.xlsx')

In [5]:
agrupaciones = ['Muy_Alto_riesgo_cLDL_controlado',
       'Alto_riesgo_cLDL_controlado',
       'Muy_Alto_riesgo_HDL_controlado',
       'Alto_riesgo_HDL_controlado',
       'Trigliceridos_controlados']
variable = 'Hipolipemiante'

In [6]:
edad_intervals = [0, 49, 59, 79, 150]
edad_labels = ['<50', '50-59', '60-79', '>80']

# Creación de grupo por intervalos de edad
df['edad_grupo'] = pd.cut(
    df['edad_calculada'], bins=edad_intervals, labels=edad_labels, right=False
)

for agrupacion in agrupaciones:
    # Crear tabla de frecuencias cruzadas con totales incluidos
    tabla_frecuencias = pd.crosstab(
        df[agrupacion], df[variable], margins=True, margins_name="Total"
    )
    
    # Reordenar las columnas en el orden 'si', 'no', 'Total'
    if 'si' in tabla_frecuencias.columns and 'no' in tabla_frecuencias.columns:
        tabla_frecuencias = tabla_frecuencias[['si', 'no', 'Total']]
    
    # Calcular porcentajes por columnas
    totales_columna = tabla_frecuencias.loc["Total", :]
    porcentajes = (tabla_frecuencias.div(totales_columna, axis=1) * 100).round(2)
    
    # Combinar frecuencias y porcentajes en un formato "valor (porcentaje%)"
    tabla_final_percent = tabla_frecuencias.astype(str) + " (" + porcentajes.astype(str) + "%)"
    
    # Crear título dinámico para la esquina superior izquierda
    titulo_esquina = f"{agrupacion}/{variable}"
    
    # Crear tabla visual con plotly
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=[titulo_esquina] + list(tabla_final_percent.columns),
            fill_color='paleturquoise',
            align='center'
        ),
        cells=dict(
            values=[list(tabla_final_percent.index)] + [tabla_final_percent[col].values for col in tabla_final_percent.columns],
            fill_color='lavender',
            align='center'
        )
    )])
    
    # Ajustar diseño de la tabla
    fig.update_layout(
        title=f"Tabla de Frecuencias: {agrupacion} vs {variable}"
    )
    fig.show()

In [8]:
import pandas as pd
import plotly.graph_objects as go

# Definición de intervalos
hb1a_interv = {'POC_hba1c(%)': [0, 5.7, 6.4, 7.5, 10, max(df['POC_hba1c(%)']) + 4]}
pres_sist = {'masculino': [0, 120, 130, 139, max(df['presion_arterial_sistolica'])],
             'femenino': [0, 120, 130, 139, max(df['presion_arterial_sistolica'])]}
IMC_interv = {'IMC': [0, 19.9, 24.9, 29.9, 40, max(df['IMC']) + 1]}
per_inter = {'masculino': [0, 94, 102, max(df['perimetro_abdominal'])],
             'femenino': [0, 80, 88, max(df['perimetro_abdominal'])]}
pres_diast = {'masculino': [0, 80, 90, 120, max(df['presion_arterial_diastolica'])],
              'femenino': [0, 80, 90, 120, max(df['presion_arterial_diastolica'])]}

# Creación de intervalos en el DataFrame
df['POC_hba1c_intervalo'] = pd.cut(df['POC_hba1c(%)'], bins=hb1a_interv['POC_hba1c(%)'])
df['IMC_intervalo'] = pd.cut(df['IMC'], bins=IMC_interv['IMC'])
df['perimetro_abdominal_intervalo'] = pd.cut(df['perimetro_abdominal'], bins=per_inter['masculino'])
df['presion_arterial_sistolica_intervalo'] = pd.cut(df['presion_arterial_sistolica'], bins=pres_sist['masculino'])
df['presion_arterial_diastolica_intervalo'] = pd.cut(df['presion_arterial_diastolica'], bins=pres_diast['masculino'])
variables = ['IMC_intervalo', 'POC_hba1c_intervalo', 'perimetro_abdominal_intervalo', 
             'presion_arterial_sistolica_intervalo', 'presion_arterial_diastolica_intervalo']

for variable in variables:
    for agrupacion in agrupaciones:
        # Crear tabla de frecuencias cruzadas
        tabla_frecuencias = pd.crosstab(df[agrupacion], df[variable])

        # Calcular totales por columna
        totales_columna = tabla_frecuencias.sum(axis=0)

        # Calcular porcentajes por columna correctamente
        porcentajes = tabla_frecuencias.div(totales_columna, axis=1) * 100

        # Agregar columna de totales para las filas
        tabla_frecuencias['Total'] = tabla_frecuencias.sum(axis=1)

        # Calcular porcentaje en la columna "Total" basado en el total global
        porcentajes['Total'] = (tabla_frecuencias['Total'] / tabla_frecuencias['Total'].sum() * 100).round(2)

        # Agregar fila de totales (valores absolutos y porcentaje 100%)
        tabla_frecuencias.loc['Total'] = tabla_frecuencias.sum(axis=0)
        porcentajes.loc['Total'] = [100.0] * len(tabla_frecuencias.columns)

        # Combinar frecuencias y porcentajes en formato "valor (porcentaje%)"
        tabla_final_percent = tabla_frecuencias.astype(str) + " (" + porcentajes.round(2).astype(str) + "%)"

        # Reordenar columnas si es necesario
        columnas_ordenadas = ['si', 'no', 'Total']
        columnas_actuales = [col for col in columnas_ordenadas if col in tabla_final_percent.columns]
        tabla_final_percent = tabla_final_percent[columnas_actuales]

        # Crear tabla visual con Plotly
        fig = go.Figure(data=[go.Table(
            header=dict(
                values=['Categoría'] + list(tabla_final_percent.columns),
                fill_color='paleturquoise',
                align='center'
            ),
            cells=dict(
                values=[tabla_final_percent.index] + [tabla_final_percent[col].values for col in tabla_final_percent.columns],
                fill_color='lavender',
                align='center'
            )
        )])

        # Título dinámico de la tabla
        fig.update_layout(
            title=f'Tabla de Frecuencias: {agrupacion} vs {variable}'
        )
        fig.show()

In [9]:

variables = ['IMC', 'POC_hba1c(%)', 'perimetro_abdominal', 'presion_arterial_sistolica', 'presion_arterial_diastolica']

# Iterar sobre cada combinación de agrupación y variable
for agrupacion in agrupaciones:
    for variable in variables:
        # Agrupar la variable en función de los intervalos definidos
        if variable == 'IMC':
            df[variable + '_grupo'] = pd.cut(df[variable], IMC_interv['IMC'])
        elif variable == 'POC_hba1c(%)':
            df[variable + '_grupo'] = pd.cut(df[variable], hb1a_interv['POC_hba1c(%)'])
        elif variable == 'presion_arterial_sistolica':
            if agrupacion == 'masculino':
                df[variable + '_grupo'] = pd.cut(df[variable], pres_sist['masculino'])
            else:
                df[variable + '_grupo'] = pd.cut(df[variable], pres_sist['femenino'])
        elif variable == 'perimetro_abdominal':
            if agrupacion == 'masculino':
                df[variable + '_grupo'] = pd.cut(df[variable], per_inter['masculino'])
            else:
                df[variable + '_grupo'] = pd.cut(df[variable], per_inter['femenino'])
        elif variable == 'presion_arterial_diastolica':
            if agrupacion == 'masculino':
                df[variable + '_grupo'] = pd.cut(df[variable], pres_diast['masculino'])
            else:
                df[variable + '_grupo'] = pd.cut(df[variable], pres_diast['femenino'])

        df_grouped = df.groupby([agrupacion, variable + '_grupo']).size().reset_index(name='counts')
        fig_treemap = px.treemap(df_grouped, path=[agrupacion, variable + '_grupo'], values='counts',
                                 title=f'Treemap de {agrupacion} vs {variable}')
        fig_treemap.show()
        fig_barras_dobles = px.bar(df_grouped, x=agrupacion, y='counts', color=variable + '_grupo',
                                   title=f'Barras Dobles de {agrupacion} vs {variable}', barmode='group')
        fig_barras_dobles.show()
        fig_barras_apiladas = px.bar(df_grouped, x=agrupacion, y='counts', color=variable + '_grupo',
                                     title=f'Barras Apiladas de {agrupacion} vs {variable}', barmode='stack')
        fig_barras_apiladas.show()

variables = ['IMC', 'POC_hba1c(%)', 'perimetro_abdominal', 'presion_arterial_sistolica', 'presion_arterial_diastolica']

variables = ['IMC', 'POC_hba1c(%)', 'perimetro_abdominal', 'presion_arterial_sistolica', 'presion_arterial_diastolica']

for agrupacion in agrupaciones:
    for variable in variables:
        if variable == 'IMC':
            df[variable + '_grupo'] = pd.cut(df[variable], IMC_interv['IMC'])
        elif variable == 'POC_hba1c(%)':
            df[variable + '_grupo'] = pd.cut(df[variable], hb1a_interv['POC_hba1c(%)'])
        elif variable == 'presion_arterial_sistolica':
            if agrupacion == 'masculino':
                df[variable + '_grupo'] = pd.cut(df[variable], pres_sist['masculino'])
            else:
                df[variable + '_grupo'] = pd.cut(df[variable], pres_sist['femenino'])
        elif variable == 'perimetro_abdominal':
            if agrupacion == 'masculino':
                df[variable + '_grupo'] = pd.cut(df[variable], per_inter['masculino'])
            else:
                df[variable + '_grupo'] = pd.cut(df[variable], per_inter['femenino'])
        elif variable == 'presion_arterial_diastolica':
            if agrupacion == 'masculino':
                df[variable + '_grupo'] = pd.cut(df[variable], pres_diast['masculino'])
            else:
                df[variable + '_grupo'] = pd.cut(df[variable], pres_diast['femenino'])
        df_grouped = df.groupby([agrupacion, variable + '_grupo']).size().reset_index(name='counts')
        fig_treemap = px.treemap(df_grouped, path=[agrupacion, variable + '_grupo'], values='counts',
                                 title=f'Treemap de {agrupacion} vs {variable}')
        fig_treemap.show()
        fig_barras_dobles = px.bar(df_grouped, x=agrupacion, y='counts', color=variable + '_grupo',
                                   title=f'Barras Dobles de {agrupacion} vs {variable}', barmode='group')
        fig_barras_dobles.show()
        fig_barras_apiladas = px.bar(df_grouped, x=agrupacion, y='counts', color=variable + '_grupo',
                                     title=f'Barras Apiladas de {agrupacion} vs {variable}', barmode='stack')
        fig_barras_apiladas.show()